In [1]:
# === SETUP: Run this first! ===
import os
import sys

# Change to project root and add to Python path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)  # Goes up one level from 'notebooks/'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"tsnn module path: {os.path.join(project_root, 'tsnn')}")

Project root: /Users/gremy/Code/TSNN-1
tsnn module path: /Users/gremy/Code/TSNN-1/tsnn


In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import importlib
import sys
#sys.path.append('/Users/cyrilgarcia/notebooks/tsnn/')

import tsnn

from tsnn.generators import generators
from tsnn.benchmarks import benchmark_comparison, ml_benchmarks, torch_benchmarks
from tsnn import utils
import torch.nn.functional as F
import math
from typing import Optional
from tsnn.tstorch import transformers



plt.style.use('ggplot')

In [3]:
from dataclasses import dataclass
from torch import nn
device = 'mps'

In [4]:
from typing import Dict

In [5]:
from tsnn.tstorch import models

In [6]:
from tsnn.tstorch.models import GlobalMLP, BiDimensionalMLP, OneDimensionalTransformer, CustomBiDimensionalTransformer


# Summary

In [7]:
# In this notebook we will run the experiments to generate the figures for the paper.

# Test dataset

In [8]:
# We work with the following data.

In [9]:
# Global parameters
T_max = 5000
N1 = 10
F1 = 20
T1 = 5 # This parameter will be the n_rolling

print(T_max, N1, F1, T1)

5000 10 20 5


In [10]:
def generate_synthetic_datasets(
    num_time_steps: int = 3000,
    num_time_series: int = 10,
    num_features: int = 10,
    low_corr: float = 0.1,
    high_corr: float = 0.2,
    pct_zero_corr: float = 0.5,
) -> Dict[str, "generators.Generator"]:
    """
    Generates 5 synthetic multivariate time series datasets with different
    types of cross-series dependencies.

    Returns
    -------
    dict
        Keys: "d_lin", "d_cond", "d_shift", "d_cs", "d_cs_shift", "d_all"
        Values: generators.Generator objects (already with .train and .test)
    """
    dic_data = {}

    # Helper to avoid repeating the same 10 lines
    def make_gen(split_conditional=0.0,
                 split_shift=0.0,
                 split_seasonal=0.0,
                 split_cs=0.0,
                 split_cs_shift=0.0):
        gen = generators.Generator(num_time_steps, num_time_series, num_features)
        gen.generate_dataset(
            pct_zero_corr=pct_zero_corr,
            split_conditional=split_conditional,
            split_shift=split_shift,
            split_seasonal=split_seasonal,
            split_cs=split_cs,
            split_cs_shift=split_cs_shift,
            low_corr=low_corr,
            high_corr=high_corr,
        )
        return gen

    dic_data["d_lin"] = make_gen()

    # 1. Pure conditional (causal) dependence
    dic_data["d_cond"] = make_gen(split_conditional=1.0)

    # 2. Pure lagged (time-shifted) dependence
    dic_data["d_shift"] = make_gen(split_shift=1.0)

    # 3. Pure contemporaneous cross-sectional correlation
    dic_data["d_cs"] = make_gen(split_cs=1.0)

    # 4. Contemporaneous + lagged cross-series
    dic_data["d_cs_shift"] = make_gen(split_cs_shift=1.0)

    # 5. Equal mix of all four mechanisms
    dic_data["d_all"] = make_gen(
        split_conditional=0.2,
        split_shift=0.2,
        split_cs=0.2,
        split_cs_shift=0.2,
    )

    return dic_data

In [11]:
# list_low_corr = [0.01, 0.025, 0.05, 0.1]
# list_high_corr = [2*x for x in list_low_corr]

list_low_corr = [0.01, 0.03, 0.05, 0.1, 0.3, 0.5]
list_high_corr = list_low_corr

dic_data = {}

for i in range(len(list_low_corr)):
    name = "noise" + str(list_low_corr[i])
    dic_data[name] = generate_synthetic_datasets(num_time_steps=T_max, num_time_series=N1, num_features=F1, low_corr=list_low_corr[i], high_corr=list_high_corr[i])
    

In [12]:
dic_data.keys()

dict_keys(['noise0.01', 'noise0.03', 'noise0.05', 'noise0.1', 'noise0.3', 'noise0.5'])

In [13]:
effects = list(dic_data['noise0.1'].keys())
print(effects)

['d_lin', 'd_cond', 'd_shift', 'd_cs', 'd_cs_shift', 'd_all']


In [14]:
# We will fix the above dataset for now.

In [15]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool

mask = causal_mask
mask = build_attention_mask(mask, T1, device=device)
def custom_mask_mod(b, h, q_idx, kv_idx):
    return mask[q_idx, kv_idx]

# List of models

In [16]:
# Let's list here all the models we wish to test on all the data.

In [17]:
MLP_global = GlobalMLP(N1, F1, T1, dropout=0.2).to(device)

MLP_2D = BiDimensionalMLP(N1, F1, T1, dropout=0.2).to(device)

trans_1D_T4 = OneDimensionalTransformer(N1, F1, T1, mask=mask, attn_direction="T",  num_attn_layers=4,
                                        dropout=0.2, roll_y=True).to(device)
#Note: using the MLP compression seems very bad..

trans_2D_TCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                    dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                           roll_y=True).to(device)

trans_2D_TCTCTCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                    dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                           roll_y=True).to(device)


In [18]:
dic_models = {'MLP_global':MLP_global, 'MLP_2D':MLP_2D, "trans_1D_T4":trans_1D_T4, "trans_2D_TCTC":trans_2D_TCTC, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC}

In [19]:
trans_2D_TCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                    dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                           roll_y=False).to(device)

trans_2D_TCTCTCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                    dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                           roll_y=False).to(device)


In [20]:
dic_models_rollfalse = {"trans_2D_TCTC":trans_2D_TCTC_rollfalse, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC_rollfalse}

In [21]:
# For each model we also need to specify the option we will use to fit them.

In [22]:
dic_data['noise0.1']

{'d_lin': <tsnn.generators.generators.Generator at 0x173566c10>,
 'd_cond': <tsnn.generators.generators.Generator at 0x173e76f10>,
 'd_shift': <tsnn.generators.generators.Generator at 0x16915ee90>,
 'd_cs': <tsnn.generators.generators.Generator at 0x173267190>,
 'd_cs_shift': <tsnn.generators.generators.Generator at 0x173cff5d0>,
 'd_all': <tsnn.generators.generators.Generator at 0x173cff910>}

In [23]:
dic_data.keys()

dict_keys(['noise0.01', 'noise0.03', 'noise0.05', 'noise0.1', 'noise0.3', 'noise0.5'])

In [24]:
list(dic_models.keys())

['MLP_global', 'MLP_2D', 'trans_1D_T4', 'trans_2D_TCTC', 'trans_2D_TCTCTCTC']

# Function to create table for an effect

In [36]:
# We give the function that creates for a given effect the table testing all models and all noise level.

In [27]:
def run_models(effect1):

    # Storage
    records_train = []
    records_test  = []

    for noise_level in dic_data.keys():          
        for model_key in dic_models.keys():                   
            print(f"Running → {noise_level} | {model_key}")

            z = dic_data[noise_level][effect1]
            if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1, roll_y=True)
            else:
                z.get_dataloader(n_rolling=T1)

            if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                lr=0.001/2
            else:
                lr=0.001

            epochs = 20
            if noise_level in ['noise0.01', 'noise0.03']:
                epochs = 40

            model = dic_models[model_key]

            # Model
            if noise_level == 'noise0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1)
                model = dic_models_rollfalse[model_key]
                epochs = 60
                lr=0.0001

            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer)

            
            # Train
            wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

            # Compare
            comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

            corr_train = comp.correl(z, mode="train", return_values=True)
            corr_test  = comp.correl(z, mode="test",  return_values=True)

            train_corr = corr_train.loc["model1", "optimal"]
            test_corr  = corr_test.loc["model1", "optimal"]

            # Save results
            records_train.append({
                "noise_level": noise_level,
                "model": model_key,
                "train_corr_optimal": train_corr
            })
            records_test.append({
                "noise_level": noise_level,
                "model": model_key,
                "test_corr_optimal": test_corr
            })

    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    # Pivot: rows = noise level, columns = effect type
    train_pivot = df_train.pivot(index="noise_level", columns="model", values="train_corr_optimal")
    test_pivot  = df_test.pivot(index="noise_level", columns="model", values="test_corr_optimal")

    col_order = list(dic_models.keys())
    train_pivot = train_pivot[col_order]
    test_pivot  = test_pivot[col_order]

    return train_pivot, test_pivot

## Running on linear effect

In [28]:
table_train_lin0, table_test_lin0 = run_models('d_lin')

Running → noise0.01 | MLP_global


100%|██████████| 40/40 [00:07<00:00,  5.71it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → noise0.01 | MLP_2D


100%|██████████| 40/40 [00:10<00:00,  3.99it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → noise0.01 | trans_1D_T4


...

100%|██████████| 20/20 [00:08<00:00,  2.29it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → noise0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:23<00:00,  1.19s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → noise0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:44<00:00,  2.23s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [29]:
display(table_train_lin0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,
noise0.01,0.032,0.032,0.036,0.283,0.148
noise0.03,0.084,0.086,0.089,0.098,0.093
noise0.05,0.152,0.162,0.167,0.217,0.168
noise0.1,0.300,0.334,0.324,0.365,0.337
noise0.3,0.697,0.779,0.731,0.818,0.749
noise0.5,0.851,0.903,0.872,0.930,0.886


In [30]:
display(table_test_lin0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,
noise0.01,0.038,0.026,0.048,0.290,0.138
noise0.03,0.098,0.066,0.125,0.111,0.111
noise0.05,0.102,0.134,0.247,0.282,0.238
noise0.1,0.118,0.355,0.436,0.436,0.485
noise0.3,0.566,0.861,0.831,0.858,0.840
noise0.5,0.857,0.933,0.928,0.943,0.929


In [32]:
# Saving the data

#table_train_lin0.to_csv('table_train_lin.csv', index=True)
#table_test_lin0.to_csv('table_test_lin.csv', index=True)

In [ ]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on conditional effect

In [33]:
table_train_cond0, table_test_cond0 = run_models('d_cond')

Running → noise0.01 | MLP_global


100%|██████████| 40/40 [00:07<00:00,  5.32it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → noise0.01 | MLP_2D


100%|██████████| 40/40 [00:11<00:00,  3.55it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → noise0.01 | trans_1D_T4


...

100%|██████████| 20/20 [00:08<00:00,  2.35it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → noise0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:23<00:00,  1.19s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → noise0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:44<00:00,  2.21s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [34]:
display(table_train_cond0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,
noise0.01,0.038,0.039,0.038,0.088,0.066
noise0.03,0.085,0.081,0.087,0.090,0.084
noise0.05,0.154,0.142,0.155,0.174,0.160
noise0.1,0.297,0.266,0.300,0.331,0.306
noise0.3,0.657,0.592,0.672,0.731,0.699
noise0.5,0.798,0.705,0.829,0.902,0.873


In [35]:
display(table_test_cond0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,
noise0.01,0.001,-0.013,0.018,0.071,0.045
noise0.03,0.004,0.002,-0.008,0.069,0.072
noise0.05,-0.006,-0.007,0.010,0.122,0.122
noise0.1,0.018,0.026,0.016,0.251,0.236
noise0.3,-0.001,0.005,0.024,0.655,0.653
noise0.5,-0.005,0.009,0.033,0.851,0.835


In [ ]:
# Saving the data

#table_train_lin0.to_csv('table_train_lin.csv', index=True)
#table_test_lin0.to_csv('table_test_lin.csv', index=True)

In [ ]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on shift effect

In [ ]:
table_train_shift0, table_test_shift0 = run_models('d_shift')

In [ ]:
display(table_train_shift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,
noise0.01,0.032,0.032,0.036,0.283,0.148
noise0.03,0.084,0.086,0.089,0.098,0.093
noise0.05,0.152,0.162,0.167,0.217,0.168
noise0.1,0.300,0.334,0.324,0.365,0.337
noise0.3,0.697,0.779,0.731,0.818,0.749
noise0.5,0.851,0.903,0.872,0.930,0.886


In [ ]:
display(table_test_shift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,
noise0.01,0.038,0.026,0.048,0.290,0.138
noise0.03,0.098,0.066,0.125,0.111,0.111
noise0.05,0.102,0.134,0.247,0.282,0.238
noise0.1,0.118,0.355,0.436,0.436,0.485
noise0.3,0.566,0.861,0.831,0.858,0.840
noise0.5,0.857,0.933,0.928,0.943,0.929


In [ ]:
# Saving the data

#table_train_lin0.to_csv('table_train_lin.csv', index=True)
#table_test_lin0.to_csv('table_test_lin.csv', index=True)

In [ ]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs effect

In [ ]:
table_train_cs0, table_test_cs0 = run_models('d_cs')

In [ ]:
display(table_train_cs0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,
noise0.01,0.032,0.032,0.036,0.283,0.148
noise0.03,0.084,0.086,0.089,0.098,0.093
noise0.05,0.152,0.162,0.167,0.217,0.168
noise0.1,0.300,0.334,0.324,0.365,0.337
noise0.3,0.697,0.779,0.731,0.818,0.749
noise0.5,0.851,0.903,0.872,0.930,0.886


In [ ]:
display(table_test_cs0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,
noise0.01,0.038,0.026,0.048,0.290,0.138
noise0.03,0.098,0.066,0.125,0.111,0.111
noise0.05,0.102,0.134,0.247,0.282,0.238
noise0.1,0.118,0.355,0.436,0.436,0.485
noise0.3,0.566,0.861,0.831,0.858,0.840
noise0.5,0.857,0.933,0.928,0.943,0.929


In [ ]:
# Saving the data

#table_train_lin0.to_csv('table_train_lin.csv', index=True)
#table_test_lin0.to_csv('table_test_lin.csv', index=True)

In [ ]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs_shift

In [ ]:
table_train_csshift0, table_test_csshift0 = run_models('d_cs_shift')

In [ ]:
display(table_train_csshift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,
noise0.01,0.032,0.032,0.036,0.283,0.148
noise0.03,0.084,0.086,0.089,0.098,0.093
noise0.05,0.152,0.162,0.167,0.217,0.168
noise0.1,0.300,0.334,0.324,0.365,0.337
noise0.3,0.697,0.779,0.731,0.818,0.749
noise0.5,0.851,0.903,0.872,0.930,0.886


In [ ]:
display(table_test_csshift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,
noise0.01,0.038,0.026,0.048,0.290,0.138
noise0.03,0.098,0.066,0.125,0.111,0.111
noise0.05,0.102,0.134,0.247,0.282,0.238
noise0.1,0.118,0.355,0.436,0.436,0.485
noise0.3,0.566,0.861,0.831,0.858,0.840
noise0.5,0.857,0.933,0.928,0.943,0.929


In [ ]:
# Saving the data

#table_train_lin0.to_csv('table_train_lin.csv', index=True)
#table_test_lin0.to_csv('table_test_lin.csv', index=True)

In [ ]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

# Function to create table for all effects

In [ ]:
def run_models_all_effect(noise_level1):

    # Storage
    records_train = []
    records_test  = []

    for noise_level in dic_data.keys():          
        for model_key in dic_models.keys():                   
            print(f"Running → {noise_level} | {model_key}")

            z = dic_data[noise_level1][effect1]
            if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1, roll_y=True)
            else:
                z.get_dataloader(n_rolling=T1)

            if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                lr=0.001/2
            else:
                lr=0.001

            epochs = 20
            if noise_level in ['noise0.01', 'noise0.03']:
                epochs = 40

            model = dic_models[model_key]

            # Model
            if noise_level == 'noise0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1)
                model = dic_models_rollfalse[model_key]
                epochs = 60
                lr=0.0001

            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer)

            
            # Train
            wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

            # Compare
            comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

            corr_train = comp.correl(z, mode="train", return_values=True)
            corr_test  = comp.correl(z, mode="test",  return_values=True)

            train_corr = corr_train.loc["model1", "optimal"]
            test_corr  = corr_test.loc["model1", "optimal"]

            # Save results
            records_train.append({
                "noise_level": noise_level,
                "model": model_key,
                "train_corr_optimal": train_corr
            })
            records_test.append({
                "noise_level": noise_level,
                "model": model_key,
                "test_corr_optimal": test_corr
            })

    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    # Pivot: rows = noise level, columns = effect type
    train_pivot = df_train.pivot(index="noise_level", columns="model", values="train_corr_optimal")
    test_pivot  = df_test.pivot(index="noise_level", columns="model", values="test_corr_optimal")

    col_order = list(dic_models.keys())
    train_pivot = train_pivot[col_order]
    test_pivot  = test_pivot[col_order]

    return train_pivot, test_pivot